#### COLIEE 2024

This is the implementation of Task 1 of the 2024 Competition on Legal Information and Extraction/Entailment (COLIEE) by Damian Curran and Mike Conway.

Details of the implementation can be found in our paper 'Similarity Ranking of Case Law Using Propositions as Features' (2024).

#### Imports

In [6]:
pip install torch spacy tqdm pandas scikit-learn transformers nltk numpy tensorflow datasets


/bin/bash: line 1: /home/user/Documents/ir/.venv/bin/python: No such file or directory
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import functions from helper python files.

import t5train_code, file_code, pairs_code, model_code
import importlib 
import pandas as pd

/home/user/Documents/ir/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-19 15:01:03.253623: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-19 15:01:03.602484: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-19 15:01:04.749660: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly diff

#### t5 Proposition Extraction Model

In [ ]:
# Fine-tune t5-base on training data

importlib.reload(t5train_code)
from t5train_code import get_trainer, train_save_model

trainer = get_trainer()
train_save_model(trainer)

Set random seed to 42
Building trainer on device: cuda:0
Training model


/home/user/Documents/ir/COLIEE_2024_Task1/t5train_code.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
100,0.940200
200,0.719900
300,0.662000
400,0.612400
500,0.642000
600,0.623900
700,0.608500
800,0.582100
900,0.591800
1000,0.555600


Saved model to ./models/trained_t5


: 

#### Files

In [2]:
importlib.reload(file_code)
from file_code import (
    get_files, add_paragraphs, get_paragraphs_formatted, add_suppressed_sections, add_propositions, get_english_propositions,
    add_sentences, get_english_sentences,
    add_quotes, add_entities, add_strings_sets, add_set_lists, add_judge_name, add_year,
    get_embeddings)

In [3]:
files = get_files()

Reading raw data from text files. Generating files dataframe.


Read-in test files: 100%|██████████| 2159/2159 [00:01<00:00, 1543.98it/s]

Returning "files" df.


In [4]:
files_all = files

In [ ]:
# row_true = files_all[files_all['query'] == True].sample(n=1)
# row_false = files_all[files_all['query'] == False].sample(n=1)

# # Combine them
# files = pd.concat([row_true, row_false])
# files

,filename,set,query,cases,text
4612,088208,train,True,"[038593.txt, 003138.txt, 082319.txt, 040526.txt]","Imperial Oil Resources v. Can. (A.G.), [DATE_S..."
1470,015268,train,False,[],[1]\n: This is an application by the defendant...


In [ ]:
# Generate files dataframe, one row per file. Extract file features using the following functions:

add_paragraphs(files)
get_paragraphs_formatted(files)
add_suppressed_sections(files)
add_propositions(files)
files["propositions_en"] = files["propositions"]
add_sentences(files)
files["sentences_en"] = files["sentences"]
files
add_quotes(files)
add_entities(files)
add_strings_sets(files)
add_set_lists(files)
add_judge_name(files)
add_year(files)

Reading text from files. Extracting paragraphs based on regex pattern.


Extracting all paragraphs: 100%|██████████| 9289/9289 [00:05<00:00, 1763.26it/s]

Updated "files" df has with "paragraphs".
Getting formatted paragraphs of length < 250 words.



Getting formatted paragraphs: 100%|██████████| 9289/9289 [00:06<00:00, 1389.99it/s]

Added formatted paragraphs to "files" df in "paragraphs_formatted".
Using regex and spacy (for long paragraphs) to extract and modify suppressed sections from paragraphs:



Extracting suppressed sections from paragraphs: 100%|██████████| 9289/9289 [08:34<00:00, 18.05it/s]  

Added suppressed sections to "suppressed_sections" field.
Using pre-trained t5 model to extract propositions from suppressed sections:



The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Generating propositions from suppressed sections:  31%|███▏      | 2903/9289 [11:12<3:07:02,  1.76s/it]

In [ ]:
files.to_csv('final/embeddings_new_2_all_final_bef_emb.csv', index=False)
files

In [ ]:

get_embeddings(files)

In [ ]:
# Save as pickle (preserves lists, dicts, etc.)
import os 
save_dir="prop_one"
pickle_path = os.path.join(save_dir, "embeddings_new_2_without_prop_afterprop_mpnet.pkl")
files.to_pickle(pickle_path)


In [13]:
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,...,sentences_en_string,sentences_en_set,sentences_en_set_list,paragraphs_formatted_en_set_list,propositions_en_set_list,judge,year,embeddings_sentences_en,embeddings_paragraphs_formatted,embeddings_propositions_en
4612,088208,train,True,"[038593.txt, 003138.txt, 082319.txt, 040526.txt]","Imperial Oil Resources v. Can. (A.G.), [DATE_S...",[[1] : This is an application by Imperial Oil ...,[: This is an application by Imperial Oil Reso...,"[[13] This Court has interpreted the word ""tit...",[In exercising the powers under section 52 of ...,[In exercising the powers under section 52 of ...,...,applic imperi oil resourc imperi oil exxonmobi...,"{kaminski, expung, understand, integr, name, p...","[{record, research, relat, canadian, imperi, u...","[{record, research, relat, canadian, imperi, u...","[{accomplish, substitut, test, court, subsect,...",None,2011,"[[0.12356927, 0.09963425, -0.0032166762, 0.112...","[[0.12356927, 0.09963425, -0.0032166762, 0.112...","[[0.17937672, -0.17482978, 0.030258114, -0.030..."
1470,015268,train,False,[],[1]\n: This is an application by the defendant...,[[1] : This is an application by the defendant...,[: This is an application by the defendant in ...,[],[],[],...,applic defend patent infring action order stri...,"{belong, system, base, understand, specifi, ap...","[{fact, ground, infring, allegedli, paragraph,...","[{fact, ground, infring, allegedli, paragraph,...",[],None,2024,"[[0.095430866, -0.13481575, 0.06998002, 0.0552...","[[0.09543091, -0.1348157, 0.069979936, 0.05525...",[]


In [ ]:

files.to_csv('embeddings_all_orig.csv', index=False)


#### Pairs

In [ ]:
importlib.reload(pairs_code)
from pairs_code import (get_pairs, add_bins, get_prop_max_cos_sim_sents, get_prop_max_cos_sim_paras,
                        get_prop_max_jaccard_sents, get_prop_max_jaccard_paras, get_prop_max_overlap_sents, get_prop_max_overlap_paras, add_max_overall,
                        get_case_jaccard_sims, check_same_case, get_case_tfidf_scores, get_num_quotes, binarize_quotes, check_years, add_judge_checks)

In [ ]:
# Generate pairs dataframe. One query-candidate case pair per row. Compare file features from files df to generate pair features:

pairs = get_pairs(files)
get_prop_max_cos_sim_sents(files, pairs)
get_prop_max_cos_sim_paras(files, pairs)
get_prop_max_jaccard_sents(files,pairs)
get_prop_max_jaccard_paras(files,pairs)
get_prop_max_overlap_sents(files,pairs)
get_prop_max_overlap_paras(files,pairs)
add_max_overall(pairs,files)
get_case_jaccard_sims(files,pairs)
check_same_case(pairs)
get_case_tfidf_scores(files,pairs)
get_num_quotes(files,pairs)
binarize_quotes(pairs)
check_years(files,pairs)
add_judge_checks(files,pairs)
add_bins(files, pairs)

#### Model

In [ ]:
# Do k-fold validation on train set to identify best hyperparameters:

importlib.reload(model_code)
from model_code import get_k_fold_model_dev_pairs, save_model_df_pairs

model_df_pairs = get_k_fold_model_dev_pairs(pairs)
save_model_df_pairs(model_df_pairs)

In [ ]:
importlib.reload(model_code)
from model_code import apply_models_to_dfs
apply_models_to_dfs(model_df_pairs, infer_type=1)

In [ ]:
importlib.reload(model_code)
from model_code import apply_models_to_dfs
apply_models_to_dfs(model_df_pairs, infer_type=2)

#### Final Inference

In [1]:
# Train model

importlib.reload(model_code)
from model_code import build_train_model

train_df = pairs[pairs['set']=='train']
model, train_df = build_train_model(train_df)

In [2]:
# Generate final results

importlib.reload(model_code)
from model_code import inference_on_test

test_df = pairs[pairs['set']=='test']

for infer_type in [1,2]:
    results_df = inference_on_test(model, test_df, infer_type)
    print()

In [ ]:
import pickle

# Replace 'your_file.pkl' with your actual pickle file path
with open('/media/user/FBED-354D/ir/embeddings_new_2_without_prop_afterpropparasent.pkl', 'rb') as file:
    data = pickle.load(file)


KeyError: 0

In [2]:

print(data)

      filename    set  query cases  \
0        79311  train  False    []   
1        15880  train  False    []   
2        42979  train  False    []   
3        79847  train  False    []   
4        76089  train  False    []   
...        ...    ...    ...   ...   
9284     87439   test   True    []   
9285     56791   test  False    []   
9286     63269   test  False    []   
9287     39286   test  False    []   
9288     80537   test  False    []   

                                                   text  \
0     Strayer, J.\n: These are two applications unde...   
1     Federal Court\nFebruary 22, 2008.  <FRAGMENT_S...   
2     Federal Court\nAngus Grant, for the applicant;...   
3     MLB unedited judgment  <FRAGMENT_SUPPRESSED>\n...   
4     Federal Court\nJanuary 18, 2010.  <FRAGMENT_SU...   
...                                                 ...   
9284  [1]\nO'Reilly, J.\n: The Minister of National ...   
9285  [1]\n: When this matter first came before me, ...   
9286  [1]\n: